In [4]:
%load_ext rpy2.ipython

In [5]:
import pandas as pd

import src
from src.load import DataLoader

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [6]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /Users/lukas/git/ytpop
In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  library ‘/nix/store/4s8z8il6zyq77ixy4b8kfzwsnz90vsrm-apple-sdk-11.3/Library’ contains no packages


# Load Data

In [7]:
dl = DataLoader()

videos = dl.channels().join(dl.videos(filtered=True), "channel_id").to_pandas()
sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [8]:
channel_overview = (
    videos.merge(sents, on="video_id")
    .groupby("channel", observed=True)
    .agg(
        ch_videos=("channel", "size"),
        ch_followers=("channel_followers", "first"),
        n_sentences=("n_sents", "sum"),
        n_elite=("n_elite", "sum"),
        n_pplcentr=("n_pplcentr", "sum"),
        avg_likes=("video_likes", "mean"),
        avg_views=("video_views", "mean"),
        avg_duration=("video_duration", "mean"),
        avg_comments=("video_comments", "mean"),
        first_video=("video_uploadtime", "min"),
        latest_video=("video_uploadtime", "max"),
    )
)

In [9]:
channel_overview

,ch_videos,ch_followers,n_sentences,n_elite,n_pplcentr,avg_likes,avg_views,avg_duration,avg_comments,first_video,latest_video
channel,,,,,,,,,,,
AfD BT,5215,388000,295255,39747,5984,3861.4,45749.6,436.5,397.6,2017-12-06,2024-01-20
AfD TV,1442,250000,142810,17056,3592,3652.1,43601.4,666.1,398.4,2017-12-07,2024-01-19
CDU,613,21900,53017,918,1419,67.7,9165.2,661.3,43.1,2017-12-11,2024-01-19
CSU,141,5170,10337,286,245,36.1,22944.1,453.4,7.7,2017-12-14,2023-10-05
FDP,461,23300,36094,1403,886,0.5,5580.9,670.5,0.7,2018-01-06,2024-01-06
Greens,455,26100,45667,1415,1314,75.8,4459.9,901.3,0.2,2018-01-27,2023-12-13
Left,429,29000,45912,2420,1374,260.0,10734.1,878.3,46.3,2017-12-11,2024-01-17
SPD,469,24200,65523,1380,2163,100.5,5204.8,1133.0,25.7,2017-12-07,2024-01-18


In [10]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 1481.54 hours


In [11]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 9225


In [12]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 694615


In [13]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
ch_videos,5215.0,1442.0,613.0,141.0,461.0,455.0,429.0,469.0
ch_followers,388000.0,250000.0,21900.0,5170.0,23300.0,26100.0,29000.0,24200.0
n_sentences,295255.0,142810.0,53017.0,10337.0,36094.0,45667.0,45912.0,65523.0
n_elite,39747.0,17056.0,918.0,286.0,1403.0,1415.0,2420.0,1380.0
n_pplcentr,5984.0,3592.0,1419.0,245.0,886.0,1314.0,1374.0,2163.0
avg_likes,3861.4,3652.1,67.7,36.1,0.5,75.8,260.0,100.5
avg_views,45749.6,43601.4,9165.2,22944.1,5580.9,4459.9,10734.1,5204.8
avg_duration,436.5,666.1,661.3,453.4,670.5,901.3,878.3,1133.0
avg_comments,397.6,398.4,43.1,7.7,0.7,0.2,46.3,25.7


In [14]:
path = src.OUT / "tables/dataset_summary.csv"
summary_table.to_csv(path)

# View Count Violin Plot

In [15]:
df = videos.merge(sents, on="video_id")

In [16]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=5)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

ggsave(here(r_out, "/figures/view_count.svg"))

Saving 13.9 x 8.33 in image


# Populism Amount Plot

In [41]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

ggsave(here(r_out, "/figures/populism_per_party.svg"))

Saving 13.9 x 8.33 in image
